# Capítulo 15: Redes Neurais

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 18 de Grus (2019).

> Gosto de bobagem; ela desperta as células cerebrais.
>
> — Dr. Seuss

Uma **rede neural artificial** é um modelo preditivo inspirado — de longe, e com muita licença poética — no funcionamento do cérebro. Pense no cérebro como uma coleção de neurônios ligados uns aos outros: cada neurônio olha as saídas dos neurônios que chegam até ele, faz uma conta, e ou dispara (se a conta passa de um limiar) ou não dispara. Um neurônio artificial faz exatamente isso, com uma conta que cabe em duas linhas de Python.

Este capítulo tem uma espinha dorsal curta, e vale enxergá-la antes de começar: **cada uma das duas seções seguintes existe por causa de um limite da anterior, e a quarta gasta tudo o que elas construíram num problema só.** Um único neurônio calcula E, OU e NÃO — mas não calcula XOR, e a razão é geométrica, não uma questão de escolher melhor os pesos. Empilhar neurônios em camadas resolve o XOR, só que a essa altura já não dá para treinar a coisa: a função degrau que faz o neurônio disparar tem derivada zero em toda parte onde ela existe, e indefinida no salto, e o gradiente descendente não tem para onde ir. Trocar o degrau pela **sigmoide** conserta isso e é o que permite usar aqui a máquina construída no [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html). A **retropropagação** é o nome de como essa máquina funciona quando há camadas empilhadas. E a última seção joga tudo isso num problema absurdo, para ver a rede aprender algo que ninguém lhe explicou.

> **❗ Importante — O que muda de categoria aqui**
>
> Este é o primeiro modelo do livro cujos **parâmetros não significam nada sozinhos**.
>
> O [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) entregou coeficientes que se leem em português: mantendo tudo o mais constante, cada amigo a mais corresponde a cerca de um minuto a mais por dia no site. O [Capítulo 14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html) entregou uma árvore que um humano **lê** de cima a baixo, e a leitura *é* a explicação da previsão. Aqui, os 25 vetores de peso da camada escondida da última seção não têm leitura nenhuma. Olhar para um deles não informa nada, e olhar para os 25 informa menos ainda.
>
> E é essa rede ilegível que resolve o problema da última seção — que nenhum dos dois modelos legíveis resolveria a partir de dez bits. É esse o ponto: **poder de representação e interpretabilidade andam em direções opostas**, e a troca é deliberada, não um efeito colateral. Uma disciplina que só ensinasse a chamar `.fit()` deixaria você fazer essa troca sem perceber que a fez.

O que este capítulo entrega para o próximo: o [Capítulo 16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) pega a rede que você vai construir aqui — uma lista de listas de listas de `float`, com a retropropagação escrita à mão para exatamente duas camadas — e a generaliza numa biblioteca de camadas componíveis. Tudo o que parecer rígido demais aqui é de propósito; é o que o próximo capítulo existe para consertar.

Ao final deste capítulo, você será capaz de:

- Implementar um perceptron e explicar, geometricamente, por que ele calcula E, OU e NÃO mas não calcula XOR
- Construir uma rede feed-forward em Python puro e usá-la para calcular o XOR com pesos escolhidos à mão
- Explicar por que trocar a função degrau pela sigmoide é o que torna o gradiente descendente aplicável a uma rede
- Derivar a retropropagação como a regra da cadeia aplicada camada a camada, e implementá-la
- Treinar uma rede a partir de pesos aleatórios e identificar, nos pesos aprendidos, os atributos que a camada escondida inventou sozinha
- Recodificar um problema que não é numérico como vetores de entrada e de saída
- Explicar o que se troca, em interpretabilidade, ao ganhar poder de representação

## Seções

| Seção | Tópico |
|---|---|
| [15.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/01-perceptrons.html) | Perceptrons |
| [15.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/02-redes-feed-forward.html) | Redes Neurais Feed-Forward |
| [15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html) | Retropropagação |
| [15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html) | Exemplo: Fizz Buzz |

## Perceptrons

> **📌 Nota**
>
> Esta seção corresponde a *Perceptrons*, do capítulo 18 de Grus (2019).

A rede neural mais simples possível tem um neurônio só, e esse neurônio se chama **perceptron**. Ele recebe $n$ entradas, calcula uma soma ponderada delas e "dispara" — devolve 1 — se essa soma for maior ou igual a zero. Caso contrário, devolve 0.

São duas funções:

In [ ]:
from scratch.linear_algebra import Vector, dot

def step_function(x: float) -> float:
    return 1.0 if x >= 0 else 0.0

def perceptron_output(weights: Vector, bias: float, x: Vector) -> float:
    """Devolve 1 se o perceptron 'dispara', 0 se não"""
    calculation = dot(weights, x) + bias
    return step_function(calculation)

O `bias` — em português, o **viés** do neurônio, que é como o texto vai chamá-lo daqui em diante — é um deslocamento: ele decide *quão grande* a soma ponderada precisa ser para o neurônio disparar. O `dot` é o produto escalar do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/index.html), e é ele que faz todo o trabalho.

Cuidado com a palavra, porque esta é a primeira vez neste livro em que "viés" não é o viés estatístico. Aqui ele é um **parâmetro** — um número que se soma à conta e que o treino ajusta como qualquer outro peso —, e não tem relação nenhuma com o erro sistemático do compromisso viés-variância da [seção 8.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-vies-e-variancia.html). O [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) tinha exatamente este parâmetro na regressão e o chamou de *termo constante*, o que evitava a colisão; a literatura de redes neurais gastou a palavra, e é ela que vamos usar.

### O perceptron é uma reta

Vale parar num detalhe que parece formalidade e não é. O perceptron devolve 1 exatamente quando

```python
dot(weights, x) + bias >= 0
```

O conjunto de pontos em que essa expressão vale **zero** é um **hiperplano** — em duas dimensões, uma reta; em três, um plano. Ele corta o espaço em duas metades, e o perceptron devolve 1 numa delas e 0 na outra.

> **🔷 Conceito**
>
> Um perceptron não é um classificador qualquer: ele é **uma reta** (ou um hiperplano, em dimensão maior). Tudo o que ele consegue fazer é dizer de que lado dessa reta um ponto está.
>
> Escolher os pesos e o viés é escolher a reta. Não há escolha de pesos que produza duas retas, nem uma curva, nem uma região no meio.

Isso já é suficiente para resolver alguns problemas. Um deles é a porta lógica **E**, que devolve 1 quando as duas entradas valem 1 e 0 em qualquer outro caso:

In [ ]:
and_weights = [2., 2]
and_bias = -3.

assert perceptron_output(and_weights, and_bias, [1, 1]) == 1
assert perceptron_output(and_weights, and_bias, [0, 1]) == 0
assert perceptron_output(and_weights, and_bias, [1, 0]) == 0
assert perceptron_output(and_weights, and_bias, [0, 0]) == 0

A aritmética é simples de conferir na mão. Se as duas entradas valem 1, a conta dá $2 + 2 - 3 = 1$, e a saída é 1. Se só uma delas vale 1, dá $2 + 0 - 3 = -1$, e a saída é 0. Se as duas valem 0, dá $-3$, e a saída é 0.

Pelo mesmo raciocínio, a porta **OU** sai movendo o viés:

In [ ]:
or_weights = [2., 2]
or_bias = -1.

assert perceptron_output(or_weights, or_bias, [1, 1]) == 1
assert perceptron_output(or_weights, or_bias, [0, 1]) == 1
assert perceptron_output(or_weights, or_bias, [1, 0]) == 1
assert perceptron_output(or_weights, or_bias, [0, 0]) == 0

Repare que os pesos são os mesmos do E; o que mudou foi só o viés, de $-3$ para $-1$. Geometricamente, isso é a mesma reta deslocada: com $-3$ ela passa entre $[1,1]$ e os outros três pontos, com $-1$ ela passa entre $[0,0]$ e os outros três.

E a porta **NÃO**, que tem uma entrada só e inverte:

In [ ]:
not_weights = [-2.]
not_bias = 1.

assert perceptron_output(not_weights, not_bias, [0]) == 1
assert perceptron_output(not_weights, not_bias, [1]) == 0

### O que um perceptron não consegue fazer

Existe uma porta lógica que nenhum perceptron calcula, por mais que você tente: o **XOR**, o "ou exclusivo", que devolve 1 quando exatamente uma das entradas vale 1, e 0 nos outros dois casos.

Desenhar os três problemas lado a lado torna a razão visível:

In [ ]:
# Figura: Os quatro pontos de entrada, para as portas E, OU e XOR. Preenchido = a porta devolve 1. Nos dois primeiros painéis, a reta é a fronteira do perceptron e a região cinza é o lado em que ele dispara — os pontos preenchidos caem todos dentro dela. O XOR não tem reta nem região cinza: nenhuma reta deixa os dois pontos preenchidos de um lado e os dois vazios do outro.
from matplotlib import pyplot as plt

pontos = [(0, 0), (0, 1), (1, 0), (1, 1)]
portas = [
    ("E",   lambda a, b: a and b,          lambda x: 1.5 - x),
    ("OU",  lambda a, b: a or b,           lambda x: 0.5 - x),
    ("XOR", lambda a, b: int(a != b),      None),
]

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6))
grade = [-0.5 + i / 50 for i in range(101)]

for ax, (nome, f, reta) in zip(axes, portas):
    if reta is not None:
        ys = [reta(x) for x in grade]
        ax.plot(grade, ys, color='0.35')
        ax.fill_between(grade, ys, 1.6, color='0.85')

    for (a, b) in pontos:
        dispara = f(a, b)
        ax.scatter([a], [b], s=130, zorder=3,
                   facecolors='black' if dispara else 'white',
                   edgecolors='black', linewidths=1.6)

    ax.set_title(nome)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.6)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

Nos dois primeiros painéis, a região cinza é o lado da reta em que o perceptron dispara, e os pontos preenchidos caem todos dentro dela. No terceiro não há região cinza porque não há reta: os dois pontos que devem devolver 1 estão em cantos **opostos** do quadrado, e os dois que devem devolver 0 estão nos outros dois cantos opostos. Qualquer reta que você desenhe deixa um ponto preenchido junto de um vazio.

O XOR não é **linearmente separável**, e essa é uma propriedade dos dados, não do algoritmo.

> **🟩 Exemplo — A impossibilidade em quatro desigualdades**
>
> O argumento geométrico convence, mas o algébrico fecha a questão em quatro linhas. Suponha que existam pesos $w_1$, $w_2$ e um viés $b$ que resolvam o XOR. Então:
>
> | entrada | saída pedida | desigualdade |
> |---|---|---|
> | $[0,0]$ | 0 | $b < 0$ |
> | $[0,1]$ | 1 | $w_2 + b \geq 0$ |
> | $[1,0]$ | 1 | $w_1 + b \geq 0$ |
> | $[1,1]$ | 0 | $w_1 + w_2 + b < 0$ |
>
> Some as duas do meio: $w_1 + w_2 + 2b \geq 0$, ou seja, $w_1 + w_2 \geq -2b$. A última linha diz que $w_1 + w_2 < -b$.
>
> Juntando as duas: precisaríamos de $-2b \leq w_1 + w_2 < -b$. Mas a primeira linha diz que $b < 0$, o que faz $-2b$ ser **maior** que $-b$. Não existe número que seja ao mesmo tempo maior ou igual a $-2b$ e menor que $-b$ quando $-2b > -b$.
>
> Não há pesos. Não é questão de procurar melhor.

Vale registrar a piada que o livro-texto faz aqui, porque ela tem conteúdo. Você não precisa aproximar um neurônio para construir uma porta lógica:

In [ ]:
and_gate = min
or_gate = max
xor_gate = lambda x, y: 0 if x == y else 1

assert and_gate(1, 1) == 1 and and_gate(0, 1) == 0
assert or_gate(0, 1) == 1 and or_gate(0, 0) == 0
assert xor_gate(1, 0) == 1 and xor_gate(1, 1) == 0

Nenhum destes três precisa de peso, viés ou treino. As portas lógicas não são o objetivo — são o menor problema em que dá para enxergar exatamente onde um neurônio sozinho para de servir. E, como os neurônios de verdade, os artificiais só ficam interessantes quando se conectam uns aos outros. É o que a [seção 15.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/02-redes-feed-forward.html) faz.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O perceptron não é só um exercício histórico: ele existe como modelo na biblioteca, com regra de treino própria.
>
> ```python
> from sklearn.linear_model import Perceptron
>
> X = [[0, 0], [0, 1], [1, 0], [1, 1]]
> y = [0, 0, 0, 1]                      # a porta E
>
> p = Perceptron(max_iter=10000, tol=None, random_state=0).fit(X, y)
> p.coef_, p.intercept_
> ```
>
> Rodando isso (`scikit-learn` 1.9), o modelo aprende a porta E com `coef_ = [[3., 2.]]` e `intercept_ = [-4.]` — pesos diferentes dos que escolhemos à mão, e igualmente corretos: são duas retas diferentes que separam os mesmos quatro pontos. Para a porta OU, o resultado é mais divertido: ele chega a `coef_ = [[2., 2.]]` e `intercept_ = [-1.]`, **exatamente** os números que o livro-texto escolheu de cabeça.
>
> E o XOR? A biblioteca não avisa nada. Ela ajusta, devolve um modelo, e com `random_state=0` esse modelo tem `coef_ = [[0., 0.]]` e `intercept_ = [0.]` — o vetor de pesos inteiramente zerado, isto é, um classificador que responde 0 para tudo. Trocando a semente, os pesos finais mudam; o que **não** muda é o resultado: nas seis primeiras sementes testadas, a acurácia deu 0,5 nas seis. Dois dos quatro pontos — exatamente o que qualquer resposta constante acerta. Não poderia ser diferente, e o argumento das quatro desigualdades acima diz por quê: não existem pesos.
>
> Repare no que aconteceu: `fit` retornou sem erro, `predict` devolveu previsões, e nada na interface indica que o problema era impossível para esta família de modelos. Nenhuma mensagem, nenhum aviso. Saber que um perceptron é uma reta — e olhar os quatro pontos antes de ajustar — é o que teria evitado a hora perdida.

## Redes Neurais Feed-Forward

> **📌 Nota**
>
> Esta seção corresponde a *Feed-Forward Neural Networks*, do capítulo 18 de Grus (2019).

A topologia do cérebro é complicadíssima, então costuma-se aproximá-la por uma idealização: a **rede feed-forward**, feita de camadas discretas de neurônios, cada uma ligada à seguinte. Há uma **camada de entrada**, que só repassa as entradas adiante sem alterá-las; uma ou mais **camadas escondidas**, cujos neurônios pegam as saídas da camada anterior, fazem uma conta e passam o resultado adiante; e uma **camada de saída**, que produz o resultado final.

Como no perceptron, cada neurônio que não é de entrada tem um peso para cada uma das suas entradas, mais um viés. Para simplificar a representação, vamos usar um truque de contabilidade: **o viés entra no fim do vetor de pesos, e o neurônio ganha uma entrada extra que vale sempre 1**. Assim o viés vira só mais um peso, e a conta do neurônio vira um produto escalar puro, sem termo solto.

### O degrau precisa sair

Aqui está a decisão mais importante do capítulo, e ela é fácil de ler como detalhe técnico quando é o contrário.

No perceptron, o neurônio somava os produtos das entradas pelos pesos e aplicava a `step_function` ao resultado. Numa rede, no lugar dela vamos aplicar uma **aproximação suave** do degrau. O motivo não é estético.

> **🔷 Conceito**
>
> Treinar uma rede significa ajustar os pesos por gradiente descendente — a máquina do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html). E gradiente descendente precisa de uma derivada que diga alguma coisa.
>
> A derivada da função degrau é **zero em toda parte onde ela existe**, e é indefinida no salto. Um gradiente identicamente nulo não aponta direção nenhuma: "andar na direção oposta ao gradiente" vira "não andar". Um gradiente indefinido não é sequer um número.
>
> Ou seja: **sobre a função degrau, nada do Capítulo 5 se aplica.** Não é que o treino fique difícil; é que o algoritmo não tem entrada válida. Trocar o degrau pela sigmoide não é um refinamento — é o que torna todo o resto deste capítulo possível.

Isso responde à pergunta que a [seção 15.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/01-perceptrons.html) deixou no ar quando disse que o `Perceptron` da biblioteca tem regra de treino própria: a regra do perceptron histórico é "a cada exemplo classificado errado, empurre os pesos na direção da entrada", e ela **não** é gradiente descendente. Existe justamente porque, sobre o degrau, o gradiente não estava disponível — e vale para um neurônio só, porque só o neurônio de saída tem um alvo contra o qual medir "errado". Com camadas escondidas, ela não tem o que dizer, e é aí que a sigmoide deixa de ser opção.

A substituta é a **sigmoide**:

In [ ]:
import math

def sigmoid(t: float) -> float:
    return 1 / (1 + math.exp(-t))

> **📌 Nota — Você já escreveu esta função**
>
> Essa é, letra por letra, a função logística do [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html) — a mesma `1 / (1 + math.exp(-x))`, construída lá por um motivo completamente diferente: espremer uma combinação linear dentro de $[0, 1]$ para que ela pudesse ser lida como probabilidade.
>
> A diferença é só de nome, e ela tem uma lógica. "Sigmoide" se refere à **forma** da curva — um S —, e há muitas funções sigmoides; "logística" se refere a **esta** função em particular. Na prática as duas palavras são usadas como sinônimos o tempo todo, inclusive por quem sabe a diferença.
>
> Vale reter que a mesma função apareceu duas vezes, por caminhos independentes: uma vez porque precisávamos de uma probabilidade, outra porque precisávamos de um degrau derivável. Ela também herda para cá o problema de saturação que o [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) documentou — para entradas muito negativas, `math.exp(-t)` estoura o alcance do `float64`.

O que a sigmoide tem que o degrau não tem fica claro olhando as duas curvas e as duas derivadas:

In [ ]:
# Figura: À esquerda, o degrau e a sua aproximação suave, em função de $t$ — a soma ponderada que entra no neurônio. À direita, as derivadas das duas em relação a $t$.
from matplotlib import pyplot as plt

grade = [-6 + i / 50 for i in range(601)]

fig, (esq, dire) = plt.subplots(1, 2, figsize=(10.5, 3.8))

esq.plot([x for x in grade if x < 0], [0.0 for x in grade if x < 0],
         color='0.35', label='degrau')
esq.plot([x for x in grade if x >= 0], [1.0 for x in grade if x >= 0],
         color='0.35')
esq.plot(grade, [sigmoid(x) for x in grade], color='C0', label='sigmoide')
esq.set_title("as funções")
esq.set_ylim(-0.15, 1.15)
esq.set_xlabel("$t$")
esq.set_ylabel("saída do neurônio")
esq.legend(loc='lower right')

dire.plot([x for x in grade if x < 0], [0.0 for x in grade if x < 0],
         color='0.35', label='derivada do degrau')
dire.plot([x for x in grade if x > 0], [0.0 for x in grade if x > 0],
         color='0.35')
dire.plot(grade, [sigmoid(x) * (1 - sigmoid(x)) for x in grade],
         color='C0', label='derivada da sigmoide')
dire.scatter([0], [0], s=60, facecolors='white', edgecolors='0.35',
            zorder=3, linewidths=1.6)
dire.annotate("indefinida", xy=(0, 0), xytext=(1.1, 0.09),
             fontsize=9, color='0.35',
             arrowprops=dict(arrowstyle='->', color='0.35'))
dire.set_title("as derivadas")
dire.set_ylim(-0.05, 0.3)
dire.set_xlabel("$t$")
dire.set_ylabel("derivada em relação a $t$")
dire.legend(loc='upper right', prop={'size': 8})

plt.tight_layout()
plt.show()

À esquerda, a sigmoide faz o mesmo trabalho do degrau — perto de 0 para entradas bem negativas, perto de 1 para entradas bem positivas — só que atravessando o meio do caminho em vez de saltar. À direita está a razão de tudo: a derivada do degrau é a reta zerada, com um buraco na origem, enquanto a da sigmoide é um sino que chega a 0,25 em $t = 0$ e nunca é exatamente zero. É essa curva azul da direita que o gradiente descendente vai usar para saber para onde andar. A cinza não serve para nada.

Com a sigmoide no lugar, a saída de um neurônio fica assim:

In [ ]:
from scratch.linear_algebra import Vector, dot

def neuron_output(weights: Vector, inputs: Vector) -> float:
    # weights inclui o termo de viés, inputs inclui um 1
    return sigmoid(dot(weights, inputs))

### Como representar uma rede

Com essa função, um neurônio é simplesmente um **vetor de pesos**, de comprimento igual ao número de entradas mais um (por causa do viés). Uma camada é uma **lista de neurônios**. E uma rede é uma **lista de camadas** — as que não são de entrada, já que a camada de entrada não faz conta nenhuma.

Ou seja: uma rede neural é uma lista (camadas) de listas (neurônios) de vetores (pesos). Nenhuma classe, nenhum objeto; três níveis de lista.

Usar a rede, dada essa representação, é curto:

In [ ]:
from typing import List

def feed_forward(neural_network: List[List[Vector]],
                 input_vector: Vector) -> List[Vector]:
    """
    Passa o vetor de entrada pela rede.
    Devolve as saídas de todas as camadas (não só a última).
    """
    outputs: List[Vector] = []

    for layer in neural_network:
        input_with_bias = input_vector + [1]               # acrescenta a constante
        output = [neuron_output(neuron, input_with_bias)   # calcula a saída
                  for neuron in layer]                     # de cada neurônio
        outputs.append(output)                             # guarda o resultado

        # a entrada da próxima camada é a saída desta
        input_vector = output

    return outputs

Repare em duas decisões dentro do laço. A primeira é o `input_vector + [1]`, que materializa o truque do viés a cada camada. A segunda é que a função devolve as saídas de **todas** as camadas, e não só a última. Isso parece desperdício agora — quem usa a rede quer só a resposta final. Não é: a [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html) depende de ter as saídas intermediárias guardadas, e é por isso que elas são devolvidas.

### O XOR, agora sim

Com duas camadas, dá para construir à mão a porta que um perceptron sozinho não calcula. O único cuidado é usar pesos grandes, para que as saídas da sigmoide fiquem bem perto de 0 ou bem perto de 1:

In [ ]:
xor_network = [# camada escondida
               [[20., 20, -30],      # neurônio 'e'
                [20., 20, -10]],     # neurônio 'ou'
               # camada de saída
               [[-60., 60, -30]]]    # neurônio '2ª entrada mas não a 1ª'

# feed_forward devolve as saídas de todas as camadas, então [-1] pega a
# saída final, e [0] pega o valor de dentro do vetor resultante
assert 0.000 < feed_forward(xor_network, [0, 0])[-1][0] < 0.001
assert 0.999 < feed_forward(xor_network, [1, 0])[-1][0] < 1.000
assert 0.999 < feed_forward(xor_network, [0, 1])[-1][0] < 1.000
assert 0.000 < feed_forward(xor_network, [1, 1])[-1][0] < 0.001

Os `assert` passam, mas eles só dizem que a rede acerta. O interessante é *como*. Como `feed_forward` devolve todas as camadas, dá para abrir a rede no meio e ver o que a camada escondida está calculando:

In [ ]:
for entrada in [[0, 0], [0, 1], [1, 0], [1, 1]]:
    escondida, saida = feed_forward(xor_network, entrada)
    print(f"{entrada} -> escondida "
          f"[{escondida[0]:.4f}, {escondida[1]:.4f}]"
          f"   saída {saida[0]:.4f}")

Olhe a coluna do meio, arredondando na cabeça. O **primeiro** neurônio escondido só se aproxima de 1 na última linha: ele calcula o **E** das duas entradas. O **segundo** se aproxima de 1 nas três últimas: ele calcula o **OU**. A camada escondida transformou o par de entradas num par `(e, ou)`.

E o neurônio de saída, com pesos $[-60, 60, -30]$, aplica $-60 \cdot e + 60 \cdot ou - 30$: ele dispara quando o *ou* vale 1 e o *e* vale 0. Isto é, "ou, mas não e" — que é exatamente o XOR.

> **🔷 Conceito**
>
> Uma forma sugestiva de ler isso: **a camada escondida calcula atributos dos dados de entrada** — aqui, o "e" e o "ou" —, e a camada de saída combina esses atributos para gerar a resposta desejada.
>
> É a mesma ideia da extração de atributos do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html), com uma diferença que muda tudo: lá quem inventava os atributos era você. Aqui a rede vai inventá-los sozinha, assim que aprendermos a treiná-la.

Só que nós escolhemos esses pesos à mão, olhando para o problema e raciocinando sobre ele. Isso funciona para um XOR com dois neurônios escondidos, e não funciona para mais nada. A [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html) tira essa tarefa das nossas mãos.

> **💡 Dica — Na prática: `scikit-learn`**
>
> A rede acima é uma lista de listas de listas de `float`. Vale ver o que uma biblioteca guarda no lugar disso, porque as diferenças são justamente as decisões que tomamos sem discutir.
>
> ```python
> from sklearn.neural_network import MLPClassifier
>
> rede = MLPClassifier(hidden_layer_sizes=(2,), activation='logistic',
>                      solver='lbfgs', random_state=0, max_iter=5000)
> rede.fit([[0, 0], [0, 1], [1, 0], [1, 1]], [0, 1, 1, 0])
>
> rede.coefs_        # pesos, uma matriz por camada
> rede.intercepts_   # vieses, um vetor por camada
> ```
>
> Rodando isso (`scikit-learn` 1.9), `coefs_` tem duas matrizes, de formatos $(2, 2)$ e $(2, 1)$, e `intercepts_` tem dois vetores, de formatos $(2,)$ e $(1,)$. Três diferenças em relação ao nosso código, todas informativas:
>
> **O viés é guardado à parte.** A biblioteca não anexa o viés ao vetor de pesos nem acrescenta uma entrada constante 1. Aquilo era contabilidade nossa, para transformar a conta do neurônio num produto escalar limpo — não uma verdade sobre redes neurais. As duas representações descrevem a mesma rede.
>
> **A camada é uma matriz, não uma lista de neurônios.** Onde temos uma lista de vetores, a biblioteca tem uma matriz $(\text{entradas} \times \text{neurônios})$, porque a passagem por uma camada é uma multiplicação de matriz — e é assim que ela roda rápido. O nosso `for neuron in layer` faz a mesma conta, um neurônio por vez.
>
> **A ativação é um parâmetro.** Escrevemos `activation='logistic'` para pedir a nossa sigmoide; o padrão do `MLPClassifier` é `'relu'`, outra função inteiramente. Na nossa rede, a sigmoide está soldada dentro de `neuron_output` e não há como trocá-la sem editar a função. É uma das rigidezes que o [Capítulo 16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) desfaz, transformando a ativação numa camada como qualquer outra.
>
> O que a biblioteca **não** faz diferente é a conta. `dot(weights, inputs)` seguido de sigmoide é literalmente o que acontece lá dentro.

## Retropropagação

> **📌 Nota**
>
> Esta seção corresponde a *Backpropagation*, do capítulo 18 de Grus (2019).

A rede XOR da [seção 15.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/02-redes-feed-forward.html) foi construída à mão: olhamos para o problema, percebemos que "ou, mas não e" resolve, e escrevemos os pesos que produzem isso. Esse método não escala por dois motivos independentes. O primeiro é tamanho — um problema de reconhecimento de imagens envolve centenas ou milhares de neurônios, e ninguém escreve mil vetores de peso à mão. O segundo é pior: na maioria dos problemas **não há como raciocinar** o que cada neurônio deveria calcular. "E" e "ou" eram atributos óbvios do XOR; não existe o equivalente óbvio para a maior parte das tarefas.

Então, como sempre neste livro, usamos dados. O algoritmo se chama **retropropagação**, e ele é gradiente descendente — o do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html), sem substituições — aplicado a uma função cujos parâmetros estão distribuídos em camadas.

O ciclo tem cinco passos. Dado um conjunto de treino com vetores de entrada e os vetores de saída desejados:

1. Rodar `feed_forward` sobre um vetor de entrada, produzindo as saídas de **todos** os neurônios da rede.
2. Como conhecemos a saída desejada, calcular uma perda: a soma dos erros ao quadrado.
3. Calcular o gradiente dessa perda em relação aos pesos dos neurônios de **saída**.
4. "Propagar" os gradientes e os erros **para trás**, obtendo os gradientes em relação aos pesos dos neurônios **escondidos**.
5. Dar um passo de gradiente descendente.

Repetido sobre o conjunto inteiro, muitas vezes, até a rede convergir.

O passo 4 é o coração da coisa, e é onde vale gastar o tempo.

### O que significa propagar o erro para trás

O problema é este: o neurônio de saída **tem** um alvo, então dá para dizer o quanto ele errou. Um neurônio escondido não tem alvo nenhum. Ninguém nunca disse a ele qual deveria ser a sua saída. Como atribuir culpa a quem não tem gabarito?

A resposta é a regra da cadeia, e ela tem uma leitura direta. Chame de **culpa** de um neurônio a quantidade

$$
\delta \;=\; \frac{\partial \text{perda}}{\partial z}
$$

onde $z$ é a soma ponderada que entra na sigmoide daquele neurônio — o quanto a perda mudaria se empurrássemos $z$ um tiquinho para cima. Três observações, em ordem:

**Para um neurônio de saída, a culpa é direta.** A perda depende da saída $o$ por $(o - t)^2$, e a saída depende de $z$ pela sigmoide. Pela regra da cadeia, a culpa é o produto de dois fatores com significados distintos: $(o - t)$, o quanto o neurônio errou e para que lado; e $o(1-o)$, a derivada da sigmoide, que mede o quanto aquele neurônio **consegue reagir** a um empurrão. Um neurônio saturado — saída perto de 0 ou perto de 1 — tem $o(1-o)$ perto de zero e recebe pouca culpa mesmo estando muito errado, porque mexer nele quase não muda a saída.

**A culpa de um peso é a culpa do neurônio vezes a entrada que aquele peso multiplica.** Faz sentido: se a entrada que chega por um peso valia 0 naquele exemplo, aquele peso não teve influência nenhuma no resultado, e o gradiente dele é zero. Um peso só é responsável na medida em que a entrada dele estava ligada.

**A culpa de um neurônio escondido é emprestada dos que vêm depois dele.** Ele não tem alvo, mas a saída dele foi enviada a cada neurônio da camada seguinte, cada um por uma conexão de peso conhecido. Então cada neurônio de saída devolve para trás a sua própria culpa, multiplicada pelo peso da conexão por onde o sinal passou. Somando essas parcelas e multiplicando pela derivada da sigmoide do próprio neurônio escondido, temos a culpa dele.

> **🔷 Conceito**
>
> É daí que vem o nome. **O mesmo peso que levou o sinal para a frente traz a culpa de volta.** Uma conexão forte transporta muita responsabilidade; uma conexão fraca, pouca. O erro medido na saída é redistribuído para trás, camada a camada, na proporção exata em que cada peso contribuiu para produzi-lo.

### O código

Traduzido, isso é uma função só. O `feed_forward` vem importado de `scratch.neural_networks` — é o mesmo código da [seção 15.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/02-redes-feed-forward.html), não uma segunda versão dele. Cada página deste livro roda num kernel próprio, então nomes definidos numa página não existem em outra; o pacote `scratch`, copiado literalmente do repositório do livro-texto, existe em todas.

In [ ]:
from typing import List
from scratch.linear_algebra import Vector, dot
from scratch.neural_networks import feed_forward

def sqerror_gradients(network: List[List[Vector]],
                      input_vector: Vector,
                      target_vector: Vector) -> List[List[Vector]]:
    """
    Dada uma rede, um vetor de entrada e um vetor-alvo, faz uma previsão
    e calcula o gradiente da perda quadrática em relação aos pesos.
    """
    # passada para a frente
    hidden_outputs, outputs = feed_forward(network, input_vector)

    # gradientes em relação às pré-ativações dos neurônios de saída
    output_deltas = [output * (1 - output) * (output - target)
                     for output, target in zip(outputs, target_vector)]

    # gradientes em relação aos pesos dos neurônios de saída
    output_grads = [[output_deltas[i] * hidden_output
                     for hidden_output in hidden_outputs + [1]]
                    for i, output_neuron in enumerate(network[-1])]

    # gradientes em relação às pré-ativações dos neurônios escondidos
    hidden_deltas = [hidden_output * (1 - hidden_output) *
                         dot(output_deltas, [n[i] for n in network[-1]])
                     for i, hidden_output in enumerate(hidden_outputs)]

    # gradientes em relação aos pesos dos neurônios escondidos
    hidden_grads = [[hidden_deltas[i] * input for input in input_vector + [1]]
                    for i, hidden_neuron in enumerate(network[0])]

    return [hidden_grads, output_grads]

As quatro seções da função são, na ordem, os três parágrafos acima mais a passada para a frente. `output_deltas` é a culpa dos neurônios de saída; `output_grads` distribui essa culpa pelos pesos, multiplicando pela entrada de cada um (`hidden_outputs + [1]`, com o 1 do viés); `hidden_deltas` é o empréstimo — `[n[i] for n in network[-1]]` colhe, de todos os neurônios de saída, o peso da conexão que vinha do $i$-ésimo escondido; e `hidden_grads` distribui essa culpa pelos pesos da primeira camada.

> **🟩 Exemplo — A conta, para quem quiser vê-la**
>
> O livro-texto deixa a derivação como exercício. Ela cabe aqui, e não é difícil — só exige cuidado com os índices.
>
> **Cuidado com as letras.** Desta caixa em diante, $i$ numera os neurônios de **saída** e $h$, os **escondidos**. O código não faz essa distinção: ele reaproveita a letra `i` nos dois papéis, porque cada compreensão de lista é um escopo local e ali não há ambiguidade. Em `output_grads`, o `i` de `output_deltas[i]` enumera os neurônios de **saída** — é o $i$ desta caixa. Em `hidden_deltas` e em `hidden_grads`, o `i` enumera os **escondidos** — é o $h$. E em `[n[i] for n in network[-1]]` os dois papéis convivem na mesma linha: `n` percorre os neurônios de saída (o $i$ daqui) e `n[i]` indexa, dentro de cada um, o peso da conexão que vem do escondido de índice `i` (o $h$ daqui). A matemática precisa de duas letras; o código não, e é essa a fonte da confusão.
>
> Escreva $z_h$ e $o_h = \sigma(z_h)$ para a pré-ativação e a saída do $h$-ésimo neurônio escondido, e $z_i$, $o_i$ para os do $i$-ésimo neurônio de saída, com alvo $t_i$. A perda de um exemplo é
>
> $$
> L = \sum_i (o_i - t_i)^2
> $$
>
> Para o neurônio de saída $i$, a regra da cadeia dá
>
> $$
> \frac{\partial L}{\partial z_i}
> = \underbrace{\frac{\partial L}{\partial o_i}}_{2(o_i - t_i)} \cdot
>   \underbrace{\frac{\partial o_i}{\partial z_i}}_{o_i(1 - o_i)}
> $$
>
> usando que $\sigma' = \sigma(1-\sigma)$ — a propriedade conveniente da logística que o [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) já tinha registrado. Como $z_i = \sum_h v_{ih}\,o_h + v_{i,\text{viés}}$, a derivada em relação a um peso é
>
> $$
> \frac{\partial L}{\partial v_{ih}} = \frac{\partial L}{\partial z_i} \cdot o_h
> $$
>
> que é a culpa do neurônio vezes a entrada que aquele peso multiplica.
>
> Para o neurônio escondido $h$, a diferença é que $o_h$ afeta a perda por **vários** caminhos — um por neurônio de saída — e as contribuições se somam:
>
> $$
> \frac{\partial L}{\partial o_h} = \sum_i \frac{\partial L}{\partial z_i} \cdot v_{ih}
> \qquad\Longrightarrow\qquad
> \frac{\partial L}{\partial z_h} = o_h(1 - o_h) \sum_i \frac{\partial L}{\partial z_i} \cdot v_{ih}
> $$
>
> que é exatamente a linha `hidden_deltas` do código, com $v_{ih}$ escrito como `n[i]` — lembrando que ali o `i` do código é o $h$ desta caixa, e o $i$ daqui é quem `n` percorre. E, de novo, a derivada em relação a um peso da primeira camada é essa culpa vezes a entrada correspondente, $x_j$.

> **⚠️ Atenção — O fator 2 que sumiu**
>
> Compare a derivação com o código e falta um 2. A conta dá $\partial L / \partial o_i = 2(o_i - t_i)$; `output_deltas` calcula `output * (1 - output) * (output - target)`, sem o 2.
>
> Não é um erro, mas o comentário do código também não é preciso ao dizer "o gradiente da perda quadrática": o que está sendo calculado é o gradiente exato da **metade** da soma dos quadrados. Como os dois diferem por um fator constante, eles apontam exatamente na mesma direção, e o efeito prático é só o de dobrar ou não a taxa de aprendizado — que é um número escolhido à mão de qualquer forma, como o [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/04-escolhendo-o-tamanho-do-passo.html) mostrou.
>
> Vale saber disso por um motivo concreto: a [seção 15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html) vai **imprimir** a perda a cada epoch, usando `squared_distance`, que é a soma dos quadrados inteira. O número acompanhado na tela é, portanto, o dobro daquele cujo gradiente está sendo descido. Para ver a perda cair, dá no mesmo; para conferir uma conta contra a outra, não.

> **❗ Importante — Esta implementação só funciona para duas camadas**
>
> Olhe a primeira linha do corpo da função: `hidden_outputs, outputs = feed_forward(...)`. Ela desempacota a lista de saídas em exatamente **dois** nomes.
>
> Uma rede com três camadas faria essa linha estourar com `ValueError`, e uma com uma camada só também. `network[0]` e `network[-1]` aparecem no resto do código pelo mesmo motivo: são "a primeira" e "a última", e a função supõe que não há nada entre elas.
>
> Isso não é descuido; é o custo de escrever a retropropagação à mão para uma arquitetura específica. Cada nova topologia exigiria uma nova função. O [Capítulo 16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) existe em grande parte para resolver isso: lá, cada camada saberá propagar o gradiente através de si mesma, e redes de qualquer profundidade se montam encaixando camadas.

### Treinando o XOR

Vamos aprender a rede XOR que a seção anterior escreveu à mão. Começamos com os dados de treino — quatro exemplos, o problema inteiro — e uma rede de pesos aleatórios:

In [ ]:
import random

random.seed(0)

# dados de treino
xs = [[0., 0], [0., 1], [1., 0], [1., 1]]
ys = [[0.], [1.], [1.], [0.]]

# começamos com pesos aleatórios
network = [ # camada escondida: 2 entradas -> 2 saídas
            [[random.random() for _ in range(2 + 1)],   # 1º neurônio escondido
             [random.random() for _ in range(2 + 1)]],  # 2º neurônio escondido
            # camada de saída: 2 entradas -> 1 saída
            [[random.random() for _ in range(2 + 1)]]   # 1º neurônio de saída
          ]

[[round(w, 4) for w in neuron] for layer in network for neuron in layer]

Nove números entre 0 e 1: três para cada um dos dois neurônios escondidos e três para o de saída, contando o viés de cada um. É só isso que a rede sabe antes de ver o primeiro exemplo. Guarde a ordem de grandeza — ao fim desta seção, esses mesmos nove números estarão nas unidades e nas dezenas, e vários terão trocado de sinal.

O `random.seed(0)` é a regra da casa, como em todo chunk aleatório deste livro. Mas aqui ele pesa mais do que em outros capítulos: o treino de uma rede depende do ponto de partida de um jeito que uma regressão não depende, e duas inicializações diferentes podem levar a **soluções diferentes**, não só a arredondamentos diferentes — a [seção 15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html) mostra um caso em que a rede fica presa por dezenas de epochs antes de escapar.

Antes do treino, vale ver uma única passada para trás, sobre um exemplo só. A entrada $[1, 0]$ deveria produzir 1; a rede aleatória produz outra coisa, e daí sai a culpa de cada neurônio:

In [ ]:
hidden_outputs, outputs = feed_forward(network, [1., 0])

output_deltas = [o * (1 - o) * (o - t) for o, t in zip(outputs, [1.])]
hidden_deltas = [h * (1 - h) * dot(output_deltas, [n[i] for n in network[-1]])
                 for i, h in enumerate(hidden_outputs)]

print(f"saída       {outputs[0]:.4f}  (alvo 1)         culpa {output_deltas[0]:+.6f}")
for i, (h, d) in enumerate(zip(hidden_outputs, hidden_deltas)):
    print(f"escondido {i+1}  {h:.4f}   peso p/ a saída {network[-1][0][i]:.4f}"
          f"   culpa {d:+.6f}")

Três coisas para ver aí, e é para vê-las que vale interromper o texto. A primeira: as três culpas são **negativas**, porque a rede respondeu abaixo do alvo, e é o mesmo erro sendo redistribuído — o sinal não se inverte no caminho de volta. A segunda: as culpas escondidas são uma ordem de grandeza menores que a da saída. Elas são o produto da culpa de trás por um peso menor que 1 e pela derivada da sigmoide, que vale no máximo 0,25 — a culpa **encolhe** a cada camada que atravessa, e essa é a versão em miniatura do problema que trava redes profundas.

A terceira é a proporcionalidade da metáfora: o primeiro escondido manda o sinal à saída por um peso de cerca de 0,78 e o segundo por um de cerca de 0,30 — dois e meio para um —, e o primeiro recebe cerca do dobro da culpa do segundo. Não exatamente dois e meio, porque cada um ainda multiplica pela **própria** derivada, e o segundo, mais perto do meio da sigmoide, reage mais a um empurrão. Conexão forte, muita responsabilidade; conexão fraca, pouca — moderado pelo quanto cada neurônio consegue reagir.

O treino é gradiente descendente **estocástico**: o laço interno percorre os exemplos um a um e dá um passo em cada, que é a variante batizada na [seção 5.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/06-minibatch-e-estocastico.html). Vinte mil epochs sobre quatro exemplos são, portanto, 80.000 passos de gradiente, não 20.000. A outra diferença em relação aos capítulos anteriores é que agora há **vários** vetores de parâmetros, cada um com o seu gradiente, então `gradient_step` é chamado uma vez por neurônio a cada passo:

In [ ]:
from scratch.gradient_descent import gradient_step
import tqdm

learning_rate = 1.0

for epoch in tqdm.trange(20000, desc="rede neural para o xor"):
    for x, y in zip(xs, ys):
        gradients = sqerror_gradients(network, x, y)

        # um passo de gradiente para cada neurônio de cada camada
        network = [[gradient_step(neuron, grad, -learning_rate)
                    for neuron, grad in zip(layer, layer_grad)]
                   for layer, layer_grad in zip(network, gradients)]

# conferimos que ela aprendeu o XOR
assert feed_forward(network, [0, 0])[-1][0] < 0.01
assert feed_forward(network, [0, 1])[-1][0] > 0.99
assert feed_forward(network, [1, 0])[-1][0] > 0.99
assert feed_forward(network, [1, 1])[-1][0] < 0.01

Vinte mil epochs sobre quatro exemplos custam menos de um segundo. O `tqdm.trange` desenha a barra de progresso do [Capítulo 7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/07-um-parenteses-tqdm.html), e o chunk leva `#| warning: false` para que a barra não vaze para a página — ela é útil no terminal, ilegível num livro.

Os `assert` passaram, então a rede aprendeu. Vale ver com quanta folga:

In [ ]:
for x in xs:
    print(f"{x} -> {feed_forward(network, x)[-1][0]:.6f}")

> **🟩 Exemplo — A simetria não é acidente**
>
> Olhe os dois valores do meio: **0,992329** e **0,992328**. Eles concordam até a quinta casa decimal, e isso não é coincidência numérica.
>
> O XOR é **simétrico** nos seus dois argumentos: trocar a primeira entrada pela segunda não muda a resposta. Os dados de treino não distinguem $[0,1]$ de $[1,0]$ de forma nenhuma, e o gradiente que chega ao peso da primeira entrada é praticamente o espelho do que chega ao peso da segunda — seria exatamente o espelho se os dois pesos já fossem iguais. Como a inicialização foi aleatória, os dois pesos não começaram iguais — mas foram empurrados quase igualmente durante 20.000 epochs, e a diferença inicial praticamente se dissolveu.
>
> Cuidado com o que a simetria **não** diz. Ela relaciona $[0,1]$ com $[1,0]$, porque um é a troca do outro. Já $[0,0]$ e $[1,1]$ são cada um a troca de si mesmo, e nada obriga que tenham a mesma saída: de fato, 0,009034 e 0,007856 diferem na terceira casa. Os dois estão perto de zero porque o alvo dos dois é zero, o que é uma razão diferente.

### O que a rede aprendeu

A parte mais interessante é abrir os pesos:

In [ ]:
nomes = ["escondido 1", "escondido 2", "saída     "]
neuronios = [n for layer in network for n in layer]

for nome, neuron in zip(nomes, neuronios):
    exatos = ", ".join(f"{w:8.4f}" for w in neuron)
    arredondados = ", ".join(f"{round(w):3d}" for w in neuron)
    print(f"{nome}  [{exatos}]   ->  [{arredondados}]")

Arredondados, os pesos aprendidos são $[7, 7, -3]$, $[5, 5, -8]$ e $[11, -12, -5]$ — os mesmos três vetores que o livro-texto imprime para este experimento. Nada nos garantia isso: bastaria uma semente diferente para os números mudarem. O que faz eles baterem é que o `random.seed(0)` do livro-texto é o mesmo daqui, e o código também.

Repare também na simetria dentro de cada neurônio escondido: 6,9535 e 6,9528 no primeiro, 5,1159 e 5,1154 no segundo. É a mesma simetria da caixa acima, agora visível nos parâmetros em vez das saídas.

E o que esses neurônios calculam?

In [ ]:
for x in xs:
    escondida, saida = feed_forward(network, x)
    print(f"{x} -> escondida [{escondida[0]:.4f}, {escondida[1]:.4f}]"
          f"   saída {saida[0]:.4f}")

O primeiro neurônio escondido está perto de 1 nas três últimas linhas: ele calcula o **OU**. O segundo só chega perto de 1 na última: ele calcula o **E**. E o neurônio de saída, com pesos $[11, -12, -5]$, dispara quando o primeiro está ligado e o segundo não — "o primeiro mas não o segundo", ou seja, "ou, mas não e".

### Duas retas, e não uma

A [seção 15.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/01-perceptrons.html) desenhou exatamente estes quatro pontos e parou numa impossibilidade: nenhuma reta deixa os dois preenchidos de um lado e os dois vazios do outro. Cada neurônio escondido **é** uma reta — foi a primeira coisa que aquela seção estabeleceu —, então desenhar as duas que a rede acabou de aprender responde à pergunta que ela deixou em aberto:

In [ ]:
# Figura: As duas retas que a camada escondida aprendeu, sobre o mesmo quadrado das portas lógicas da seção 15.1. Cada reta é a fronteira de um neurônio escondido: de um lado ele fica perto de 0, do outro perto de 1. Na faixa cinza entre as duas, o 'ou' já disparou e o 'e' ainda não — é ali, e só ali, que a rede devolve 1. Preenchido = o XOR vale 1.
from matplotlib import pyplot as plt

# Confere qual escondido é qual, em vez de supor: para a entrada [0, 1],
# o "ou" dispara e o "e" não.
h_01, _ = feed_forward(network, [0., 1])
assert h_01[0] > 0.5 > h_01[1], "1º escondido é o 'ou', 2º é o 'e'"

def fronteira(neuron, xs):
    """x2 tal que w1*x1 + w2*x2 + b = 0, para cada x1 em xs."""
    w1, w2, b = neuron
    return [-(w1 * x + b) / w2 for x in xs]

grade = [-0.5 + i / 200 for i in range(401)]
reta_ou = fronteira(network[0][0], grade)
reta_e = fronteira(network[0][1], grade)

fig, ax = plt.subplots(figsize=(5.0, 4.8))

ax.fill_between(grade, reta_ou, reta_e, color='0.87')
ax.plot(grade, reta_ou, color='0.35')
ax.plot(grade, reta_e, color='0.35', linestyle='--')

for (a, b) in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    ax.scatter([a], [b], s=130, zorder=3,
               facecolors='black' if a != b else 'white',
               edgecolors='black', linewidths=1.6)

ax.text(0.5, 0.5, "a rede\ndevolve 1", ha='center', va='center',
        fontsize=8.5, color='0.2')

x_ou = 0.30
ax.annotate("escondido 1: o \"ou\"",
            xy=(x_ou, fronteira(network[0][0], [x_ou])[0]),
            xytext=(0.05, -0.40), ha='center', fontsize=9, color='0.2',
            arrowprops=dict(arrowstyle='->', color='0.35'))

x_e = 0.85
ax.annotate("escondido 2: o \"e\"",
            xy=(x_e, fronteira(network[0][1], [x_e])[0]),
            xytext=(0.95, 1.44), ha='center', fontsize=9, color='0.2',
            arrowprops=dict(arrowstyle='->', color='0.35'))

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.6)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

As duas retas cortam o quadrado em três faixas, e a leitura vai de baixo para cima:

- **Abaixo da reta cheia**, nenhum dos dois neurônios escondidos disparou. É onde mora $[0,0]$, e a rede devolve 0.
- **Na faixa cinza**, o "ou" já disparou e o "e" ainda não. É onde moram $[0,1]$ e $[1,0]$, e a rede devolve 1.
- **Acima da reta tracejada**, os dois dispararam. É onde mora $[1,1]$, e a rede devolve 0 de novo.

Repare que as duas retas saíram quase **paralelas**. Não é acaso: os dois pesos de cada neurônio escondido são praticamente iguais entre si — 6,9535 e 6,9528 no primeiro, 5,1159 e 5,1154 no segundo —, e uma reta $w x_1 + w x_2 + b = 0$ com pesos iguais tem inclinação $-1$, seja qual for $w$. O que separa uma reta da outra é só o viés, que a empurra para mais longe da origem. É exatamente a observação que a [seção 15.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/01-perceptrons.html) fez ao passar da porta E para a porta OU mexendo apenas no viés — e a rede, sem que ninguém lhe dissesse nada, redescobriu aquele par de retas.

E o neurônio de saída não faz mais nada além de **nomear a faixa do meio**. "Dispara quando o primeiro está ligado e o segundo não" é, geometricamente, "está acima da reta cheia e abaixo da tracejada".

É esse o negócio que uma camada escondida oferece, na forma mais nua que este livro consegue mostrar: **a possibilidade de recortar uma região entre duas fronteiras, em vez de ter que escolher um lado de uma só.** Um perceptron sozinho tem uma reta e dois lados, e acabou. Dois neurônios escondidos têm duas retas, e o neurônio de saída pode pedir qualquer combinação dos pedaços que elas produzem — inclusive a faixa do meio, que não é "um lado" de reta nenhuma. Mais neurônios escondidos recortam regiões mais elaboradas; mais camadas passam a recortar regiões de regiões. A [seção 15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html) vai usar 25 neurônios escondidos sobre dez dimensões, onde não há figura possível — e é justamente por isso que vale ter visto esta.

> **🔷 Conceito**
>
> A rede treinada encontrou **os mesmos dois atributos** que escolhemos à mão na seção anterior — o "e" e o "ou" — mas na **ordem trocada**: lá o primeiro neurônio escondido era o "e", aqui é o "ou".
>
> Isso é a primeira aparição de algo que a próxima seção leva ao extremo. Nada no algoritmo pediu "e" e "ou"; nada fixou qual neurônio ficaria com qual. **A rede tem várias soluções equivalentes, e a que ela encontra depende de onde a inicialização aleatória a colocou.** Trocar os dois neurônios escondidos de lugar (e trocar junto os dois pesos correspondentes na saída) produz uma rede que calcula exatamente a mesma função.
>
> Neste exemplo minúsculo os pesos ainda contam uma história legível, porque o problema tinha dois atributos óbvios e a rede tinha exatamente dois neurônios para descobri-los. Guarde a raridade disso: já na [seção 15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html), com vinte e cinco neurônios escondidos, a leitura direta acaba — o que sobra lá é procurar estrutura com uma hipótese na mão.

> **💡 Dica — Na prática: derivar gradientes à mão acabou**
>
> Duas coisas ficam por conta da biblioteca aqui, e elas são de naturezas diferentes.
>
> A primeira é o modelo. Este mesmo experimento, com o `scikit-learn`:
>
> ```python
> from sklearn.neural_network import MLPClassifier
>
> rede = MLPClassifier(hidden_layer_sizes=(2,), activation='logistic',
>                      solver='lbfgs', random_state=0, max_iter=5000)
> rede.fit([[0, 0], [0, 1], [1, 0], [1, 1]], [0, 1, 1, 0])
> rede.predict([[0, 0], [0, 1], [1, 0], [1, 1]])
> ```
>
> Rodando isso (`scikit-learn` 1.9), a rede acerta os quatro pontos. Repare no `solver='lbfgs'`: não é gradiente descendente simples, é um método quase-Newton que usa informação de curvatura — uma das muitas variantes que existem porque descer o gradiente puro, como fazemos aqui, costuma ser lento demais em problemas grandes.
>
> A segunda é mais profunda, e é a que mudou a área. **Ninguém deriva `sqerror_gradients` à mão hoje.** As bibliotecas de rede neural — `PyTorch`, `TensorFlow`, `JAX` — implementam **diferenciação automática em modo reverso**, que faz exatamente a contabilidade desta seção, só que genérica: cada operação elementar sabe propagar a derivada através de si mesma, e o gradiente de qualquer composição delas sai por aplicação repetida da regra da cadeia, na ordem inversa da execução. Você escreve a passada para a frente; a passada para trás é derivada do grafo de operações.
>
> É por isso que a retropropagação vale como conceito e quase nunca como código escrito à mão. E é exatamente a abstração que o [Capítulo 16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) constrói, em pequeno: uma `Layer` que sabe fazer `forward` e `backward`, e uma rede que é só uma sequência delas.

## Exemplo: Fizz Buzz

> **📌 Nota**
>
> Esta seção corresponde a *Example: Fizz Buzz*, do capítulo 18 de Grus (2019).

O vice-presidente de Engenharia da DataSciencester quer entrevistar candidatos técnicos pedindo que resolvam o "Fizz Buzz", o exercício de programação mais surrado que existe:

> Imprima os números de 1 a 100, com estas exceções: se o número for divisível por 3, imprima `"fizz"`; se for divisível por 5, imprima `"buzz"`; e se for divisível por 15, imprima `"fizzbuzz"`.

Ele está convencido de que quem resolve isso é um programador excepcional. Você acha que o problema é tão simples que uma rede neural daria conta.

O exercício, como está enunciado, transforma um inteiro numa string. Redes neurais recebem vetores e devolvem vetores. Então o primeiro trabalho — e o mais interessante desta seção — é **reformular o problema como um problema de vetores**.

### Codificando a saída

Essa parte é fácil. Há quatro respostas possíveis, então a saída vira um vetor de quatro posições, com um 1 na posição da resposta certa:

In [ ]:
from scratch.linear_algebra import Vector

def fizz_buzz_encode(x: int) -> Vector:
    if x % 15 == 0:
        return [0, 0, 0, 1]
    elif x % 5 == 0:
        return [0, 0, 1, 0]
    elif x % 3 == 0:
        return [0, 1, 0, 0]
    else:
        return [1, 0, 0, 0]

assert fizz_buzz_encode(2)  == [1, 0, 0, 0]
assert fizz_buzz_encode(6)  == [0, 1, 0, 0]
assert fizz_buzz_encode(10) == [0, 0, 1, 0]
assert fizz_buzz_encode(30) == [0, 0, 0, 1]

A ordem dos testes importa: 15 é divisível por 3 e por 5, então o caso do 15 precisa vir antes dos outros dois. É a mesma armadilha do exercício original, e ela sobreviveu à mudança de representação.

### Codificando a entrada

Essa parte é menos óbvia, e é onde está a lição.

A tentação é usar um vetor de uma posição só, contendo o próprio número. Não faça isso, por duas razões. A primeira é que uma entrada única codifica uma **intensidade**: ela diz que 2 é o dobro de 1, e que 4 é o dobro de 2 — informação verdadeira e completamente irrelevante para este problema. A segunda é que, com uma entrada só, a camada escondida não teria matéria-prima para calcular atributo interessante nenhum.

O que funciona razoavelmente bem — e o livro-texto é honesto ao dizer que isso não era óbvio nem para ele — é converter cada número para a sua **representação binária**:

In [ ]:
from typing import List

def binary_encode(x: int) -> Vector:
    binary: List[float] = []

    for i in range(10):
        binary.append(x % 2)
        x = x // 2

    return binary

#                             1  2  4  8 16 32 64 128 256 512
assert binary_encode(0)   == [0, 0, 0, 0, 0, 0, 0, 0,  0,  0]
assert binary_encode(1)   == [1, 0, 0, 0, 0, 0, 0, 0,  0,  0]
assert binary_encode(10)  == [0, 1, 0, 1, 0, 0, 0, 0,  0,  0]
assert binary_encode(101) == [1, 0, 1, 0, 0, 1, 1, 0,  0,  0]
assert binary_encode(999) == [1, 1, 1, 0, 0, 1, 1, 1,  1,  1]

Dez bits, do menos significativo para o mais significativo, o que dá conta dos números de 0 a 1023.

> **❗ Importante — Ninguém vai contar à rede o que é divisibilidade**
>
> Vale registrar o que **não** está sendo dado a ela, porque é isso que torna o resultado desta seção interessante em vez de trivial.
>
> A rede recebe dez zeros e uns. Ela não recebe o número, não recebe o resto da divisão por 3, não recebe nenhuma dica de que existe algo chamado divisibilidade. E a divisibilidade por 3 não é um bit: enquanto "par" é simplesmente o primeiro bit, "múltiplo de 3" depende de todos eles de um jeito torto — como $2^k \bmod 3$ alterna entre 1 e 2, a divisibilidade por 3 sai da soma **alternada** dos bits. A de 5 é pior: $2^k \bmod 5$ percorre o ciclo $1, 2, 4, 3$ e volta, então ela depende da posição de cada bit módulo 4.
>
> Existe regra, e ela é composta. Se a rede acertar, terá sido por ter construído sozinha alguma coisa que faz esse papel.

### Treino e teste

O objetivo é produzir as respostas para os números de **1 a 100**. Treinar nesses números seria trapaça — o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) chamaria isso de avaliar o modelo no próprio conjunto de treino, que é a forma mais rápida de se enganar sobre a qualidade dele. Então treinamos nos números de **101 a 1023**, o maior que cabe em dez bits:

In [ ]:
xs = [binary_encode(n) for n in range(101, 1024)]
ys = [fizz_buzz_encode(n) for n in range(101, 1024)]

len(xs)

São 923 exemplos de treino e 100 de teste, sem nenhuma sobreposição. É a separação mais limpa que este livro faz: não há embaralhamento, não há semente, não há como um número vazar de um lado para o outro.

A rede tem 10 neurônios de entrada (a dimensão dos vetores de entrada), 4 de saída (a dimensão dos alvos) e 25 escondidos — número escolhido por tentativa, guardado numa variável para ser fácil de mexer:

In [ ]:
import random

random.seed(0)

NUM_HIDDEN = 25

network = [
    # camada escondida: 10 entradas -> NUM_HIDDEN saídas
    [[random.random() for _ in range(10 + 1)] for _ in range(NUM_HIDDEN)],

    # camada de saída: NUM_HIDDEN entradas -> 4 saídas
    [[random.random() for _ in range(NUM_HIDDEN + 1)] for _ in range(4)]
]

sum(len(neuron) for layer in network for neuron in layer)

São 379 pesos ao todo — $25 \times 11$ na camada escondida mais $4 \times 26$ na de saída. Compare com os 4 coeficientes da regressão múltipla do [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html), que se liam um a um em português.

### Treinando

Este problema é bem maior que o XOR, e há muito mais coisa que pode dar errado, então vamos acompanhar a perda a cada epoch e guardá-la para desenhar depois. Também vamos parar em alguns epochs escolhidos para ver **o que** a rede está respondendo, e não só o quanto ela está errando:

In [ ]:
from collections import Counter
from scratch.neural_networks import feed_forward, sqerror_gradients, argmax
from scratch.gradient_descent import gradient_step
from scratch.linear_algebra import squared_distance
import tqdm

learning_rate = 1.0

perdas = []
marcos = [5, 20, 50, 100, 150, 500]
diagnostico = {}

with tqdm.trange(500) as t:
    for epoch in t:
        epoch_loss = 0.0

        for x, y in zip(xs, ys):
            predicted = feed_forward(network, x)[-1]
            epoch_loss += squared_distance(predicted, y)
            gradients = sqerror_gradients(network, x, y)

            # um passo de gradiente para cada neurônio de cada camada
            network = [[gradient_step(neuron, grad, -learning_rate)
                        for neuron, grad in zip(layer, layer_grad)]
                       for layer, layer_grad in zip(network, gradients)]

        perdas.append(epoch_loss)
        t.set_description(f"fizz buzz (perda: {epoch_loss:.2f})")

        if epoch + 1 in marcos:
            previstos = [argmax(feed_forward(network, x)[-1]) for x in xs]
            acertos = sum(1 for p, y in zip(previstos, ys) if p == argmax(y))
            diagnostico[epoch + 1] = (Counter(previstos), acertos)

Isso leva cerca de um minuto — 500 epochs sobre 923 exemplos, com uma passada para a frente e uma para trás em cada um, tudo em Python puro sobre listas. É de longe o chunk mais caro deste capítulo, e o preço é o mesmo de sempre: transparência.

Uma ressalva sobre o `epoch_loss`: como o treino é estocástico, os pesos mudam a cada um dos 923 exemplos, e a soma acumulada mistura previsões de redes ligeiramente diferentes. Ela não é a perda de uma rede fixa, e sim um resumo do epoch — bom para acompanhar a tendência, e é também o que produz os pequenos solavancos no fim da curva abaixo, quando a queda já é menor que essa mistura.

A perda caiu?

In [ ]:
# Figura: Soma dos erros ao quadrado sobre os 923 exemplos de treino, epoch a epoch
from matplotlib import pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(perdas) + 1), perdas)
plt.xlabel("epoch")
plt.ylabel("soma dos erros ao quadrado")
plt.title("Treino do Fizz Buzz")
plt.show()

Caiu — mas não do jeito liso que a palavra "convergir" sugere. Há uma queda rápida nos primeiros epochs, depois um trecho **plano e longo**, e só bem mais tarde a descida de verdade. Esse platô não é um detalhe cosmético do gráfico; ele é a parte mais instrutiva do treino, e dá para ver exatamente o que a rede está fazendo enquanto está parada nele:

In [ ]:
rotulos = ["o próprio número", "fizz", "buzz", "fizzbuzz"]

for epoch in marcos:
    contagem, acertos = diagnostico[epoch]
    dist = "  ".join(f"{rotulos[i]}: {contagem.get(i, 0)}" for i in range(4))
    print(f"epoch {epoch:3d}  acertos no treino: {acertos:3d}/923   {dist}")

Nos epochs 20 e 50, a rede responde **"o próprio número" para os 923 exemplos**, sem exceção. Ela não aprendeu nada sobre divisibilidade: descobriu apenas que essa é a resposta mais comum, e passou a dá-la sempre.

> **⚠️ Atenção — O platô é o palpite preguiçoso**
>
> Responder sempre a classe mais frequente é o classificador-base do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html): aquele contra o qual todo modelo deve ser comparado antes de se comemorar qualquer coisa. Nos dados de treino ele acerta 493 dos 923 números, ou 53,4%. E esse é exatamente o patamar em que a rede fica estacionada por dezenas de epochs.
>
> Repare que a perda não fica exatamente **parada** ali: ela continua caindo, só que na casa do décimo por epoch — entre os epochs 25 e 100 ela vai de cerca de 597 para 589, o que no gráfico é indistinguível de uma reta horizontal. O gradiente existe e aponta para o lado certo; ele é apenas minúsculo, porque o terreno é quase plano. A rede sai dali por teimosia, não por esperteza: continuamos empurrando por dezenas de epochs até a inclinação voltar.
>
> Isso tem uma consequência prática que a caixa "Na prática" no fim desta seção mostra medida: **um critério de parada razoável interrompe o treino exatamente aqui**, e devolve o palpite preguiçoso como se fosse o resultado. Quem só chama `.fit()` recebe 53% e um modelo que parece treinado.

### Resolvendo o problema

A rede devolve um vetor de quatro números, e queremos uma resposta só. A ponte é o `argmax`, o índice do maior valor:

In [ ]:
assert argmax([0, -1]) == 0               # o item 0 é o maior
assert argmax([-1, 0]) == 1               # o item 1 é o maior
assert argmax([-1, 10, 5, 20, -3]) == 3   # o item 3 é o maior

Com ele, finalmente:

In [ ]:
num_correct = 0
erros = []

for n in range(1, 101):
    x = binary_encode(n)
    predicted = argmax(feed_forward(network, x)[-1])
    actual = argmax(fizz_buzz_encode(n))
    labels = [str(n), "fizz", "buzz", "fizzbuzz"]

    if predicted == actual:
        num_correct += 1
    else:
        erros.append((n, labels[predicted], labels[actual]))

print(num_correct, "/", 100)
print("erros:", erros)

96 acertos em 100, em números que a rede **nunca viu**. O classificador-base do platô teria feito 53 — há 53 números entre 1 e 100 que não são múltiplos de 3 nem de 5 —, então os 96 são ganho real sobre o palpite preguiçoso, não sobre o vazio.

E não há sinal de sobreajuste: no conjunto de treino a rede terminou com 903 acertos em 923, isto é, 97,8%, contra 96% no teste. Menos de dois pontos separam os dois números.

Repare no que está sendo lido aí, porque é fácil ler a coisa errada. Não é o acerto no treino: acertar muito — ou até tudo — no próprio treino não é, sozinho, sinal de nada, nem bom nem ruim. O que se lê é a **queda** do treino para o teste — aqui, 1,8 ponto.

E é a queda, não a distância. A [seção 14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html) tem os dois casos lado a lado: a árvore sem poda acertava 99,7% dos exemplos com que se construiu e **caía** 17,3 pontos em dados novos, para 82,4%; a árvore de profundidade 2, **pior** no treino com 85,0%, **subia** 15,0 pontos e acertava 100% do teste. As duas diferenças têm quase o mesmo tamanho e sentidos opostos, de modo que a distância entre os dois números não distingue os dois casos — o sinal distingue. E o modelo com o melhor número de treino era o pior dos dois.

### Olhando os erros de perto

Vale olhar os erros de perto, porque eles não são todos do mesmo tipo:

In [ ]:
for n in [4, 70, 15]:
    saida = feed_forward(network, binary_encode(n))[-1]
    valores = ", ".join(f"{v:.4f}" for v in saida)
    print(f"n = {n:3d}  saída bruta [{valores}]   soma = {sum(saida):.4f}")

Para $n = 4$, a rede está genuinamente indecisa: as três primeiras saídas são 0,3389, 0,4649 e 0,2731, e "fizz" vence por pouco. É um erro de quem não sabe. Para $n = 70$, o quadro é outro: a saída "o próprio número" vale 0,9539 contra 0,0315 de "buzz". Aqui a rede está **confiantemente errada**, que é sempre o pior tipo de erro. O terceiro caso, $n = 15$, é o contraste que fecha o argumento: 0,9056 na posição de "fizzbuzz" e praticamente zero nas outras três — confiante e certa. Coloque os vetores de $n = 70$ e $n = 15$ lado a lado e nada na forma deles distingue um do outro. Só sabemos qual é qual porque temos o gabarito.

> **⚠️ Atenção — Estes quatro números não são uma distribuição de probabilidade**
>
> Olhe a coluna da soma. Para $n = 4$ ela dá 1,0769; para $n = 15$, 0,9062. Não é arredondamento: **nada** na rede força as quatro saídas a somarem 1.
>
> Cada neurônio de saída tem a sua própria sigmoide e produz um número em $[0, 1]$ independentemente dos outros. Quatro números em $[0, 1]$ não formam uma distribuição só porque estão lado a lado. Ler o 0,9539 do $n = 70$ como "95% de confiança" seria inventar uma garantia que o modelo não dá.
>
> A correção existe e tem nome — a função **softmax**, que normaliza as quatro saídas para que somem 1, acompanhada da perda de entropia cruzada em vez do erro quadrático. É o que o [Capítulo 16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) constrói, e é também o que a biblioteca faz por padrão num problema de classificação com mais de duas classes.

### O que a rede aprendeu, e por que não dá para saber

O vice-presidente de Engenharia, diante da evidência, cede e troca o desafio da entrevista por "inverta uma árvore binária". A piada é do livro-texto e é boa, mas ela não é a lição.

A lição é que **a rede construiu, a partir de dez bits, alguma coisa que faz o papel da divisibilidade por 3 e por 5 — sem que ninguém lhe dissesse que divisibilidade existe.** Ela viu 923 pares (padrão de bits, resposta) e montou, na camada escondida, alguma combinação desses bits que **funciona** — não necessariamente a soma alternada nem o ciclo módulo 4 de que a seção falou no início, apenas algo que acerta 96 dos 100.

"Faz o papel", e não "é": quem soubesse a regra não erraria 70 e 100, que são múltiplos de 5 como tantos outros que ela acertou, nem chutaria "fizz" em 4 e em 34. O que a rede tem é uma aproximação boa em 96 dos 100 casos que nunca viu — o que já é muito mais do que o palpite preguiçoso, e ainda assim não é a regra.

E aqui o capítulo cobra o preço que o índice anunciou. A [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html) abriu os pesos da rede do XOR e leu, dentro deles, um OU e um E. Vamos fazer exatamente a mesma coisa aqui, com dois dos 25 neurônios escondidos:

In [ ]:
for i in [0, 13]:
    pesos = ", ".join(f"{w:6.2f}" for w in network[0][i])
    print(f"escondido {i:2d}  [{pesos}]")

Onze números cada: os dez primeiros são os pesos dos dez bits, do menos significativo ao mais significativo, e o último é o viés. É a mesma leitura que na seção anterior devolveu "OU" e "E" — mesma estrutura de dados, mesmo código, e agora sem devolver frase nenhuma.

Quem procurar com afinco acha fragmentos. Nos dois vetores acima, os pesos dos bits 0, 4 e 8 saem quase iguais entre si: $-2{,}78$, $-2{,}69$ e $-2{,}73$ no primeiro; $-2{,}17$, $-2{,}23$ e $-2{,}29$ no segundo, enquanto os outros sete pesos de cada vetor se espalham por quase duas dezenas de unidades. Isso não é coincidência — e dá para medir o quanto não é.

#### O que dá para achar nos pesos, se você souber a pergunta

Comece pela aritmética, que é curta. Fizz buzz depende só de $n \bmod 15$, e o resíduo com que cada bit contribui para essa conta é $2^k \bmod 15$ — que **cicla com período 4**: $1, 2, 4, 8, 1, 2, 4, 8, 1, 2$. É a combinação dos dois ciclos da caixa *Ninguém vai contar à rede o que é divisibilidade*: o de período 2 da divisão por 3 e o de período 4 da divisão por 5. Consequência: **para esta tarefa, os bits 0, 4 e 8 carregam exatamente a mesma informação.** Trocar um pelo outro não muda a resposta certa. O mesmo vale para os bits 1, 5 e 9, para os bits 2 e 6, e para os bits 3 e 7.

A rede não sabe disso. Mas, se por acaso tiver chegado lá sozinha, os 25 neurônios escondidos vão dar pesos parecidos aos bits de um mesmo grupo — e isso é uma pergunta que se responde com uma conta. Para cada par de bits, a distância média entre os pesos que os 25 neurônios lhes dão, expressa como fração da escala típica de um peso:

In [ ]:
oculta = network[0]

def coluna(k):
    """Os 25 pesos que os neurônios escondidos dão ao bit k."""
    return [neuronio[k] for neuronio in oculta]

def distancia(j, k):
    """Distância média, neurônio a neurônio, entre os pesos dos bits j e k."""
    return sum(abs(a - b) for a, b in zip(coluna(j), coluna(k))) / NUM_HIDDEN

# escala típica de um peso, para as distâncias virarem fração dela
escala = sum(abs(w) for k in range(10) for w in coluna(k)) / (10 * NUM_HIDDEN)

# os bits se agrupam pelo resto que 2**k deixa na divisão por 15
grupos = {}
for k in range(10):
    grupos.setdefault(2 ** k % 15, []).append(k)

print(f"escala típica de um peso: {escala:.2f}\n")
print("bits   resíduo   distância")

for r in sorted(grupos):
    for i, j in enumerate(grupos[r]):
        for k in grupos[r][i + 1:]:
            print(f"{j}-{k}    {r:5d}    {100 * distancia(j, k) / escala:7.0f}%")

entre = [distancia(j, k) for j in range(10) for k in range(j + 1, 10)
         if 2 ** j % 15 != 2 ** k % 15]
print(f"\nos {len(entre)} pares de resíduos diferentes: "
      f"média {100 * sum(entre) / len(entre) / escala:.0f}%, "
      f"de {100 * min(entre) / escala:.0f}% a {100 * max(entre) / escala:.0f}%")

Um grupo salta da tabela. **Os bits 0, 4 e 8 estão a 2%, 4% e 3% de distância entre si** — o pior dos três pares fica mais de trinta vezes abaixo dos 138% que separam, em média, dois bits de resíduos diferentes. Ninguém contou à rede o que é divisibilidade, e ela terminou tratando como intercambiáveis exatamente os três bits que a teoria dos números diz que são intercambiáveis aqui.

E aí a história para de ser bonita, que é o que a torna interessante. No grupo seguinte, a rede achou **metade**: o par 5–9 fica a 7%, mas os pares com o bit 1 ficam a 29% e 31% — bem abaixo da referência, e ainda assim sete vezes acima do pior par do primeiro grupo. Os outros dois grupos ela **não** achou: 95% para o par 2–6 e 150% para o par 3–7 caem dentro da faixa que pares sem parentesco nenhum já ocupam, de 90% a 200%. Há pares de resíduos *diferentes* mais próximos entre si do que o par 2–6.

Uma das quatro simetrias com clareza, metade de outra, duas perdidas. É essa proporção que torna o achado crível: uma rede que tivesse encontrado as quatro seria uma anedota boa demais, e a primeira coisa a fazer com ela seria procurar o erro na medição. Esta é o que redes fazem — recuperam parte da estrutura do problema, do jeito irregular com que a descida do gradiente esbarra no que estava por perto.

Repare, por fim, no que isso **não** é. Não é interpretabilidade, e não é o mesmo que ler um coeficiente. Os pesos continuam ilegíveis um a um; nada mudou nos dois vetores impressos acima, e o que se descobriu não é *como* o neurônio combina os bits, só quais bits ele se recusa a distinguir. Foi preciso ter a hipótese antes de encontrar a evidência: sem saber que $2^k \bmod 15$ tem período 4, não haveria o que medir, e a estrutura ficaria ali, invisível. É o inverso do que o [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) oferece, onde o coeficiente já vem com o significado escrito nele.

> **🔷 Conceito**
>
> Pergunte quais atributos a camada escondida inventou, e não há resposta.
>
> São 25 vetores de 11 números cada. Nenhum deles corresponde a "é múltiplo de 3", ou a "o terceiro bit está ligado", ou a coisa alguma que se diga em português. O que a rede calcula está espalhado entre os 25, e a informação sobre divisibilidade por 3 não mora em nenhum neurônio específico — mora na combinação.
>
> Esse é o fim do arco que atravessa o livro. O [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) deu coeficientes que se leem um a um. O [Capítulo 14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html) deu uma árvore que se percorre com o dedo, e cuja leitura **é** a justificativa da previsão. Aqui os parâmetros não significam nada sozinhos — e o modelo resolve um problema que nenhum dos dois resolveria.
>
> Nenhum dos dois, e por motivos diferentes. Uma regressão sobre dez bits é uma soma ponderada de bits, e nenhuma soma ponderada de bits separa os múltiplos de 3: basta olhar 0, 1, 2 e 3, cujos dois primeiros bits são $[0,0]$, $[1,0]$, $[0,1]$ e $[1,1]$ — os dois múltiplos de 3 caem em cantos **opostos** do quadrado, que é exatamente a configuração que as quatro desigualdades da [seção 15.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/01-perceptrons.html) provaram impossível. Uma árvore, com dez atributos binários, tem dez perguntas para fazer e no fundo delas há uma folha por padrão de bits: a profundidade 10 é memorização, e nenhum dos cem padrões de 1 a 100 apareceu no treino.
>
> **Poder de representação e interpretabilidade andam em direções opostas.** Não é um defeito a ser consertado numa versão futura; é a troca que se faz ao escolher esta família de modelos. O que esta disciplina acrescenta é que você saiba que está fazendo a troca, e o que está entregando em cada direção.

O [Capítulo 16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) recebe essa rede e a desmonta em camadas que se compõem, com a softmax e a entropia cruzada da caixa acima no lugar das quatro sigmoides independentes — e aí o mesmo maquinário passa a caber num problema em que 25 neurônios escondidos não bastariam.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O mesmo experimento, com a biblioteca:
>
> ```python
> from sklearn.neural_network import MLPClassifier
>
> X_treino = [binary_encode(n) for n in range(101, 1024)]
> y_treino = [argmax(fizz_buzz_encode(n)) for n in range(101, 1024)]
> X_teste  = [binary_encode(n) for n in range(1, 101)]
> y_teste  = [argmax(fizz_buzz_encode(n)) for n in range(1, 101)]
>
> rede = MLPClassifier(hidden_layer_sizes=(25,), activation='logistic',
>                      max_iter=2000, random_state=0)
> rede.fit(X_treino, y_treino)
> rede.score(X_teste, y_teste)
> ```
>
> Essa configuração é a mais próxima da nossa: 25 neurônios escondidos, ativação logística. Rodando no container deste livro (`scikit-learn` 1.9), ela devolve **0,53** — e `rede.n_iter_` mostra por quê: o treino parou na **iteração 51**.
>
> Parou no platô. O `MLPClassifier` encerra quando a perda deixa de melhorar por `n_iter_no_change=10` iterações seguidas, com tolerância `tol=1e-4`, e o trecho quase plano do gráfico acima satisfaz esse critério com folga. O modelo devolvido é o palpite preguiçoso: acerta 53 dos 100, exatamente o classificador-base. Nenhum aviso, nenhum erro — `fit` retorna, `score` devolve um número.
>
> O nosso laço passou pelo platô porque **não tinha critério de parada nenhum**: ele roda 500 epochs e pronto. Uma decisão ingênua que, neste problema, acaba sendo a certa.
>
> Trocando as escolhas, a biblioteca aprende. Duas configurações medidas:
>
> | configuração | acerto no treino | acerto no teste | tempo |
> |---|---|---|---|
> | `(25,)`, `logistic`, `adam` (acima) | 0,534 | 0,530 | instantâneo |
> | `(100,)`, `relu`, `adam`, `max_iter=2000` | 1,000 | 0,860 | ~2 s |
> | `(25,)`, `logistic`, `solver='lbfgs'` | 1,000 | 0,850 | ~0,4 s |
> | **a nossa rede** | **0,978** | **0,960** | **~1 min** |
>
> Duas leituras honestas dessa tabela. A primeira: a biblioteca é uma a duas ordens de grandeza mais rápida, e isso é decisivo em qualquer problema real. A segunda: os 96% da nossa rede contra os 86% da melhor configuração testada **não** significam que o nosso código é melhor. São 100 números de teste, e a diferença de dez acertos cabe folgadamente dentro do ruído de escolher outra semente, outra ativação ou outro otimizador. O que a tabela mostra de fato é que a configuração muda o resultado de 53% para 86%, e que nada na interface avisa qual delas você escolheu.
>
> Um último detalhe que fecha a caixa anterior: para um problema com mais de duas classes, o `MLPClassifier` usa `softmax` na saída e entropia cruzada como perda — verificável em `rede.out_activation_` e `rede.loss`. Ele **não** faz o que fizemos aqui. Quatro sigmoides independentes treinadas com erro quadrático é a escolha do livro-texto, e é uma escolha datada; o [Capítulo 16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) constrói a alternativa moderna do zero.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 18 de Grus (2019) faz duas sugestões, e as duas são curtas.

A primeira é continuar lendo: o capítulo seguinte do livro-texto — o nosso [Capítulo 16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) — explora estes temas com muito mais profundidade, trocando a rede rígida deste capítulo por uma abstração de camadas que se compõem.

A segunda é o texto do próprio autor, [*Fizz Buzz in Tensorflow*](https://joelgrus.com/2016/05/23/fizz-buzz-in-tensorflow/), escrito em 2016 e narrado como um diálogo de entrevista de emprego. É a origem do exemplo da [seção 15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html), e vale ler depois de fazer a seção, não antes.

Para o tratamento moderno das redes neurais — inicialização de pesos, funções de ativação, regularização, arquiteturas — G{\'e}ron (2022) é o ponto de partida prático usual, e Hastie et al. (2009) traz o enquadramento estatístico do que este capítulo constrói à mão.

## Referências

- **G{\'e}ron**. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd ed.. O'Reilly Media. 2022.
- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed.. Springer. 2009.